## Cross-Country Job Volume

Software workforce size and share — first among the tracked roles, then **economy-wide** (software/IT as a share of each country's entire labor force).

In [1]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
import plotly.express as px
from sector_data import get_sector_employment, SOFTWARE_SECTOR

ROLE_LABELS = {
    'software_engineer': 'Software Engineer', 'lawyer': 'Lawyer',
    'physician': 'Physician', 'financial_analyst': 'Financial Analyst',
    'registered_nurse': 'Registered Nurse', 'civil_engineer': 'Civil Engineer',
    'construction_laborer': 'Construction Laborer', 'farm_worker': 'Farm Worker',
    'manufacturing_worker': 'Manufacturing Worker', 'retail_worker': 'Retail Worker',
}

usa = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_usa_data.csv')
india = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_india_data.csv')
china = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_china_data.csv')
df = pd.concat([usa, india, china], ignore_index=True)
vol = df[df['career_stage'] == 'mid'][['country', 'role', 'employed_thousands']].copy()
vol['role_label'] = vol['role'].map(ROLE_LABELS)
country_colors = {'USA': '#2171b5', 'India': '#31a354', 'China': '#e6550d'}

In [2]:
swe_vol = vol[vol['role'] == 'software_engineer'].copy()
fig = px.bar(
    swe_vol, x='country', y='employed_thousands', color='country',
    color_discrete_map=country_colors,
    title='Software Engineer Headcount by Country, Tracked Roles (2023, thousands)',
    labels={'employed_thousands': 'Employed (thousands)', 'country': 'Country'},
    text='employed_thousands',
)
fig.update_traces(texttemplate='%{text:,}K', textposition='outside')
fig.show()

In [3]:
usa_order = (
    vol[vol['country'] == 'USA']
    .sort_values('employed_thousands', ascending=False)['role_label'].tolist()
)
fig2 = px.bar(
    vol, x='role_label', y='employed_thousands', color='country',
    barmode='group',
    category_orders={'role_label': usa_order, 'country': ['USA', 'India', 'China']},
    color_discrete_map=country_colors,
    title='Total Employed by Tracked Role and Country (2023, thousands)',
    labels={'role_label': 'Role', 'employed_thousands': 'Employed (thousands)', 'country': 'Country'},
)
fig2.update_layout(xaxis_tickangle=-35)
fig2.show()

In [4]:
# Economy-wide: software/IT as a share of each country's ENTIRE labor force
econ = pd.concat([get_sector_employment(c) for c in ['USA', 'India', 'China']], ignore_index=True)
sw = econ[econ['sector'] == SOFTWARE_SECTOR].copy()

fig3 = px.bar(
    sw, x='country', y='pct_of_total', color='country',
    category_orders={'country': ['USA', 'India', 'China']},
    color_discrete_map=country_colors,
    title='Software / IT as % of TOTAL National Employment (2023)<br>'
          '<sup>Economy-wide — software is a far larger share of US jobs than of India\u2019s or China\u2019s</sup>',
    labels={'pct_of_total': 'Software/IT share of all jobs (%)', 'country': 'Country'},
    text='pct_of_total',
)
fig3.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig3.show()

In [5]:
fig4 = px.bar(
    sw, x='country', y='employed_millions', color='country',
    category_orders={'country': ['USA', 'India', 'China']},
    color_discrete_map=country_colors,
    title='Software / IT Workforce Size by Country (2023, millions)<br>'
          '<sup>Absolute headcount in the Information & Software/IT sector</sup>',
    labels={'employed_millions': 'Software/IT workers (millions)', 'country': 'Country'},
    text='employed_millions',
)
fig4.update_traces(texttemplate='%{text:.1f}M', textposition='outside')
fig4.show()